# LangChain Chains — Complete Tutorial
### *Connect LLM steps like LEGO blocks*

---

> **LangChain version:** 1.3.6 | **Python:** 3.11  
> **Prerequisites:** OpenAI API key, basic Python knowledge


---
## Setup

Run once to install and connect to the LLM.

In [ ]:
# !pip install -q langchain langchain-openai langchain-community openai

In [2]:
# !pip install langchain-groq

   ---------------------------------------- 0.0/137.5 kB ? eta -:--:--
   -------------------------------------- - 133.1/137.5 kB 8.2 MB/s eta 0:00:01
   ---------------------------------------- 137.5/137.5 kB 4.1 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import importlib.metadata

# List the distribution package names
packages = ["langchain", "langchain-community", "langchain-openai", "openai",
            "langchain_core", "langchain-groq",]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")

langchain version: 1.3.6
langchain-community version: 0.4.2
langchain-openai version: 1.2.2
openai version: 2.36.0
langchain_core version: 1.4.1
langchain-groq version: 1.1.3


In [3]:
import os
from dotenv import load_dotenv

load_dotenv() # read .env file

# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

True

In [4]:
import os
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# llm = ChatOpenAI(api_key=OPENAI_API_KEY, model="gpt-3.5-turbo", temperature=0.7)

GROQ_API_KEY = os.environ["GROQ_API_KEY"]
llm = ChatGroq(api_key=GROQ_API_KEY, model="llama-3.3-70b-versatile", temperature=0.7)

# Quick check
response = llm.invoke("Say hello in one word.")
print(" Connected! Test response:", response.content)

 Connected! Test response: Hello.


---
## 1️. Why Do We Need Chains?

### The Problem

Imagine you want to:
1. Take a topic from the user
2. Generate a blog post about it
3. Translate that blog post to French
4. Trim the blog to under 300 words

**Without chains**, you'd write this:

```python
# BAD : Manual, messy, hard to reuse
response1 = llm.invoke(f"Write a blog post about {topic}")
blog_post  = response1.content

response2  = llm.invoke(f"Translate this to French: {blog_post}")
translated = response2.content

response3  = llm.invoke(f"Shorten this to under 300 words {translated}")
final_text = response3.content
```

**With chains**, the same thing becomes:

```python
#  Clean, readable, reusable
result = (write_chain | translate_chain | summarize_chain).invoke({"topic": topic})
```

### What is a Chain?

> A **Chain** is a sequence of steps connected together, where the **output of one step becomes the input of the next**.

```
Input
  ↓
Step 1: Prompt Template   (formats your input)
  ↓
Step 2: LLM               (generates a response)
  ↓
Step 3: Output Parser     (cleans the response)
  ↓
Output
```

LangChain uses the `|` pipe operator to connect steps — just like Unix terminal pipes!

In [7]:
# Let's see the PROBLEM first — manual, no chains
# from langchain_openai import ChatOpenAI

# llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)

topic = "black holes"

# Step 1: manual call
response1 = llm.invoke(f"Give one interesting fact about {topic}.")
fact = response1.content

# Step 2: manual call — using output of step 1
response2 = llm.invoke(f"Make this fact sound more exciting: {fact}")
exciting_fact = response2.content

print("Original fact:")
print(fact)

print("---------------------")

print("\nExciting version:")
print(exciting_fact)
print("\n Works, but imagine doing this for 10 steps...")

Original fact:
One interesting fact about black holes is that their gravity is so strong that not even light can escape once it gets too close to the event horizon, which is the point of no return around a black hole. This is why they appear "black" to observers, as no light or radiation can escape to be seen.
---------------------

Exciting version:
Imagine being sucked into a cosmic abyss so powerful that not even the fastest thing in the universe - light - can break free from its grasp. That's the mesmerizing reality of black holes, where gravity is so incredibly strong that it warps the very fabric of space and time. Once you cross the point of no return, known as the event horizon, you're trapped forever, and not even a photon of light can escape to tell the tale. It's this mind-boggling phenomenon that renders black holes invisible to our eyes, shrouding them in an impenetrable veil of darkness, making them appear "black" to us - a haunting reminder of the unfathomable power that

---
## 2️) The `|` Pipe Operator — Your First Chain

### LCEL: LangChain Expression Language

LangChain introduced **LCEL** (LangChain Expression Language) — a clean way to build chains using the `|` pipe.

```
prompt | llm | parser
```

Reads left to right:
- `prompt` formats the input into a proper message
- `llm` sends it to the model and gets a response  
- `parser` converts the response into a clean string

### The 3 Core Building Blocks

| Block | Import | What it does |
|-------|--------|-------------|
| `ChatPromptTemplate` | `langchain_core.prompts` | Formats your input with variables |
| `ChatOpenAI` | `langchain_openai` | Calls the LLM |
| `StrOutputParser` | `langchain_core.output_parsers` | Extracts just the text string |

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Block 1: Prompt Template ───────────────────────────────────────
# {topic} is a variable — gets filled in when you invoke the chain
prompt = ChatPromptTemplate.from_template(
    "Give one interesting fact about {topic} in exactly 2 sentences."
)

# See what the prompt looks like BEFORE sending to LLM
rendered = prompt.format_messages(topic="black holes")
print("Rendered prompt:")
for msg in rendered:
    print(f"  [{msg.type}]: {msg.content}")

Rendered prompt:
  [human]: Give one interesting fact about black holes in exactly 2 sentences.


In [9]:
# ── Block 2: The Parser 
parser = StrOutputParser()

# Without parser: response is an AIMessage object
raw_response = llm.invoke(rendered)
print("Without parser:")
print(f"  Type: {type(raw_response)}")
print(f"  Value: {raw_response}")

print()

# With parser: response is a plain string
clean_response = parser.invoke(raw_response)
print("With parser:")
print(f"  Type: {type(clean_response)}")
print(f"  Value: {clean_response}")

Without parser:
  Type: <class 'langchain_core.messages.ai.AIMessage'>
  Value: content='One interesting fact about black holes is that they have a point of no return called the event horizon, where the gravitational pull is so strong that not even light can escape once it crosses the boundary. This unique characteristic makes black holes some of the most fascinating and mysterious objects in the universe, captivating the imagination of scientists and space enthusiasts alike.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 48, 'total_tokens': 116, 'completion_time': 0.170966944, 'completion_tokens_details': None, 'prompt_time': 0.002449291, 'prompt_tokens_details': None, 'queue_time': 0.052521299, 'total_time': 0.173416235}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019ed51c-9960-79c2-b4c0-

In [10]:
# ── Block 3: Connect with | to make a Chain
# This is the magic moment — three objects connected with |
chain = prompt | llm | StrOutputParser()

print("Chain type:", type(chain))
print()

# Invoke the chain — just pass the variable values
result = chain.invoke({"topic": "black holes"})

print("Result type:", type(result))   # plain string!
print("Result:", result)

Chain type: <class 'langchain_core.runnables.base.RunnableSequence'>

Result type: <class 'langchain_core.messages.base.TextAccessor'>
Result: One interesting fact about black holes is that they have a point of no return, known as the event horizon, where the gravitational pull is so strong that anything that crosses it will be trapped forever. This phenomenon is due to the incredibly dense mass of black holes, which warps the fabric of spacetime around them, creating an intense gravitational field that cannot be escaped once the event horizon is crossed.


In [11]:
# ──Advantage of above:  Reusability — same chain, different inputs 
topics = ["quantum computers", "Monte carlo Method", "topology"]

print(" Interesting Facts:\n")
for topic in topics:
    fact = chain.invoke({"topic": topic})
    print(f" {topic.title()}:")
    print(f"   {fact}")
    print()

🌍 Interesting Facts:

🔷 Quantum Computers:
   Quantum computers have the potential to solve complex problems that are currently unsolvable with traditional computers, thanks to their ability to process vast amounts of information simultaneously using quantum bits or qubits. One interesting fact about quantum computers is that they can exist in multiple states at the same time, allowing them to explore an exponentially large solution space in parallel, which could lead to breakthroughs in fields such as cryptography and optimization.

🔷 Monte Carlo Method:
   The Monte Carlo method is a computational algorithm that relies on repeated random sampling to obtain numerical results, and one interesting fact about it is that it was named after the famous Casino de Monte-Carlo in Monaco, reflecting the method's use of random chance. This method has been widely used in various fields, including physics, engineering, and finance, to solve complex problems that are difficult to solve using tradit

In [12]:
# ── Multiple variables in a prompt 
# Prompts can have as many {variables} as you need

explain_chain = (
    ChatPromptTemplate.from_template(
        "Explain {concept} to someone who is {audience}. "
        "Keep it under {sentences} sentences."
    )
    | llm
    | StrOutputParser()
)

# Invoke with all three variables
result = explain_chain.invoke({
    "concept": "machine learning",
    "audience": "a 12-year-old",
    "sentences": 3
})
print(result)

Machine learning is a way to teach computers to learn and make decisions on their own, kind of like how you learn new things in school. It works by showing the computer lots of examples of something, like pictures of dogs and cats, and then it can start to recognize patterns and make predictions, like "oh, that's a dog!" or "that's a cat!". This helps computers get better at doing tasks, like recognizing pictures or understanding what people are saying, without being explicitly told what to do.


In [13]:
# YOUR TURN
# Change the concept, audience, and sentences below and run!

result = explain_chain.invoke({
    "concept":   "the stock market",     # ← change this
    "audience":  "a curious grandparent", # ← change this
    "sentences": 3                         # ← change this
})

print(result)

The stock market is like a place where people can buy and sell small pieces of companies, kind of like buying a tiny part of your favorite store. When you buy a "stock," you're essentially betting that the company will do well and become more valuable, so you can sell your piece for a higher price later. Think of it like owning a tiny piece of a lemonade stand - if the stand becomes super popular, the value of your piece might go up, and you could sell it for a profit.


# Look at website